# Fresh vs Rotten Fruit Classification — MobileNetV2

## 1. MobileNetV2 Transfer Learning

This notebook implements MobileNetV2 transfer learning for six-class fresh-versus-rotten fruit classification.

The model uses the same verified leak-free train, validation, and test splits as the Baseline CNN so that the two models can be compared fairly.

The work in this notebook is limited to MobileNetV2 transfer learning and experimentation.

## 2. Reproducibility and Configuration

The MobileNetV2 experiments use a fixed random seed and the same image size and class order as the verified Baseline CNN.

The official dataset split is:
- Training: 9,513 images
- Validation: 2,043 images
- Test: 2,043 images

The six classes are kept in the same order throughout the project.

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

NUM_CLASSES = len(CLASS_NAMES)

print("TensorFlow version:", tf.__version__)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)

## 3. Load the Official Leak-Free Dataset Splits

The MobileNetV2 model uses the same verified train, validation, and test CSV files as the Baseline CNN.

This keeps the model comparison fair and preserves the source-group-aware split created during preprocessing.

In [ ]:
import pandas as pd
from pathlib import Path

GITHUB_RAW = (
    "https://raw.githubusercontent.com/"
    "onkar-726/fresh-rotten-fruit-classification/main/data"
)

train_df = pd.read_csv(f"{GITHUB_RAW}/train.csv")
validation_df = pd.read_csv(f"{GITHUB_RAW}/validation.csv")
test_df = pd.read_csv(f"{GITHUB_RAW}/test.csv")

print("Training rows  :", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows  :", len(test_df))
print("Total rows     :", len(train_df) + len(validation_df) + len(test_df))

print("\nColumns:")
print(train_df.columns.tolist())

## 4. Connect the Dataset Images

The CSV files contain relative image paths from the Kaggle dataset.

The actual dataset is downloaded into the current Colab runtime, and the relative paths are joined with the dataset root.

The paths are verified before building the MobileNetV2 input pipeline.

In [ ]:
import kagglehub
from pathlib import Path

# Download the Kaggle dataset into the Colab runtime.
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

print("Dataset root:", DATASET_ROOT)
print("Dataset root exists:", DATASET_ROOT.exists())

# Check that the CSV paths point to real image files.
for name, dataframe in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Testing", test_df),
]:
    full_paths = dataframe["image_path"].apply(
        lambda path: DATASET_ROOT / path
    )

    missing_count = (~full_paths.apply(Path.exists)).sum()

    print(f"{name} missing images:", missing_count)

print("\nSample image path:")
sample_relative_path = train_df.iloc[0]["image_path"]
sample_full_path = DATASET_ROOT / sample_relative_path

print("Relative:", sample_relative_path)
print("Full    :", sample_full_path)
print("Exists  :", sample_full_path.exists())

## 5. MobileNetV2 Image Preprocessing

MobileNetV2 uses its application-specific preprocessing function instead of the 0–1 normalization used by the Baseline CNN.

The preprocessing converts images to RGB, resizes them to 224 × 224 pixels, converts them to float32, and applies the MobileNetV2 preprocessing function.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def preprocess_mobilenet_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image

In [ ]:
sample_relative_path = train_df.iloc[0]["image_path"]
sample_full_path = DATASET_ROOT / sample_relative_path

processed_image = preprocess_mobilenet_image(
    tf.constant(str(sample_full_path))
)

print("Processed image shape:", processed_image.shape)
print("Processed image dtype:", processed_image.dtype)
print("Minimum value:", tf.reduce_min(processed_image).numpy())
print("Maximum value:", tf.reduce_max(processed_image).numpy())

## 6. Training-Time Data Augmentation

Training-time augmentation is used to create small variations of training images without creating additional image files on disk.

The augmentation is applied only to training data. Validation and test images remain unchanged apart from the required MobileNetV2 preprocessing.

The augmentation includes horizontal flipping, small rotations, zoom changes, brightness changes, and contrast changes.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            mode="horizontal"
        ),
        tf.keras.layers.RandomRotation(
            factor=0.05
        ),
        tf.keras.layers.RandomZoom(
            height_factor=0.10,
            width_factor=0.10
        ),
        tf.keras.layers.RandomBrightness(
            factor=0.10,
            value_range=(0, 255)
        ),
        tf.keras.layers.RandomContrast(
            factor=0.10
        ),
    ],
    name="mobileNet_training_augmentation"
)

print("Training-time augmentation created.")
print(data_augmentation.name)